# Batch 1 — zone frame exploration

Exploratory pass over the batch-1 downloads in `calib/raw/01_zones/`
(concept §3), ahead of writing the extract/transform/load step in
`calib/etl/`.

Nothing here writes to `calib/out/`: every cell either describes a file or
asserts something about it. Whatever survives scrutiny here moves into the ETL
step as settled code, and the notebook stays as the record of why it looks the
way it does.

Findings go to `calib/SOURCES.md` and `IMPLEMENTATION_DOCU.md`, not into this
notebook.

## Setup and inventory

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# Anchor on calib/ so the notebook runs from exploration/, from calib/, or
# from the repo root without edits.
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
calib = next((p for p in candidates if p.name == "calib"), None)
if calib is None:
    root = next(p for p in candidates if (p / ".git").exists())
    calib = root / "backend" / "models" / "demand" / "calib"

raw = calib / "raw" / "01_zones"
assert raw.is_dir(), f"batch-1 downloads not found at {raw}"

print(f"raw: {raw}\n")
for f in sorted(raw.rglob("*")):
    if f.is_file():
        print(f"{f.stat().st_size / 1e6:9.1f} MB  {f.relative_to(raw)}")

## NUTS 2021, level 3

The pinned vintage (concept §2.1). Distributed in EPSG:3035, so areas are
computed directly without reprojection.

In [ ]:
nuts3 = gpd.read_file(raw / "NUTS_RG_01M_2021_3035_LEVL_3.geojson")

assert nuts3.crs.to_epsg() == 3035, f"expected EPSG:3035, got {nuts3.crs}"
assert (nuts3["LEVL_CODE"] == 3).all(), "file mixes NUTS levels"

print(f"{len(nuts3)} zones\n")
print(nuts3.columns.tolist())
nuts3.head(3)

### Country coverage

`EU_STAT`, `EFTA_STAT` and `CC_STAT` ship on every feature, so membership is
read from the data rather than from a hand-maintained country list that would
drift as candidate status changes.

In [ ]:
FLAGS = {"EU_STAT": "EU", "EFTA_STAT": "EFTA", "CC_STAT": "candidate"}


def country_group(row: pd.Series) -> str:
    """Membership label from the GISCO status flags; UK carries none."""
    return next((label for col, label in FLAGS.items() if row[col] == "T"), "other")


countries = (
    nuts3.groupby("CNTR_CODE")
    .agg(zones=("NUTS_ID", "size"), **{c: (c, "first") for c in FLAGS})
    .assign(group=lambda df: df.apply(country_group, axis=1))
    .sort_values(["group", "zones"], ascending=[True, False])
)

print(countries.groupby("group")["zones"].agg(["count", "sum"]).to_string())
print()
for group, block in countries.groupby("group"):
    print(f"{group:10s} {sorted(block.index)}")

In [ ]:
print(countries[["group", "zones"]].sort_index().to_string())

### Geometry health and free covariates

`MOUNT_TYPE`, `URBN_TYPE` and `COAST_TYPE` are carried into `zones.csv` rather
than re-derived downstream. `NAME_LATN` and `NUTS_NAME` are the **region**
names (the latter in local script); `NAME_ENGL`, `NAME_FREN` and `NAME_GERM`
are **country** names joined onto each region — an easy trap when picking a
label column.

In [ ]:
nuts3["area_km2"] = nuts3.area / 1e6

print(f"invalid geometries: {(~nuts3.is_valid).sum()}")
print(f"empty geometries:   {nuts3.is_empty.sum()}")
print(f"multipart zones:    {(nuts3.geom_type == 'MultiPolygon').sum()}\n")

print(nuts3["area_km2"].describe().round(1).to_string())
print()

for col in ["URBN_TYPE", "MOUNT_TYPE", "COAST_TYPE"]:
    print(f"{col}: {nuts3[col].value_counts(dropna=False).sort_index().to_dict()}")

print("\nregion name differs from its Latin transliteration:", end=" ")
print((nuts3["NAME_LATN"] != nuts3["NUTS_NAME"]).sum())

The largest zones are where a geometric centroid is most wrong, so this list
previews how much the population weighting is buying.

In [ ]:
print(
    nuts3.nlargest(10, "area_km2")[["NUTS_ID", "NAME_LATN", "area_km2"]].to_string(
        index=False
    )
)

## Units manifest

Despite the name, `nuts-*-units.json` is not a code list with attributes: it
maps each NUTS code to the per-unit geometry files GISCO publishes for it
(5 resolutions × 3 projections, plus label points). Region names are not in
it — they come from the layer above.

It is still worth loading, because the key set is an independently
distributed authority for the expected unit count per level and per country.
Level and parent are recoverable from code width: 2 = country, 3 = NUTS-1,
4 = NUTS-2, 5 = NUTS-3.

In [ ]:
import json

units = json.loads((raw / "nuts-2021-units.json").read_text(encoding="utf-8"))

units_df = pd.DataFrame(
    {"nuts_id": code, "level": len(code) - 2, "country": code[:2]} for code in units
)

print(units_df["level"].value_counts().sort_index().to_string())
print()

u3 = set(units_df.loc[units_df["level"] == 3, "nuts_id"])
g3 = set(nuts3["NUTS_ID"])
assert u3 == g3, f"manifest/geometry mismatch: {sorted(u3 ^ g3)[:20]}"

per_country = units_df[units_df["level"] == 3].groupby("country").size()
assert per_country.equals(nuts3.groupby("CNTR_CODE").size().rename_axis("country")), (
    "per-country counts differ"
)

print(
    f"cross-check passed: {len(g3)} NUTS-3 units, no orphans, counts match per country"
)

## NUTS 2021 vs 2024

Held only to build `zone_crosswalk.csv`, so the `nuts_version` pin can move
later without re-deriving the zone frame.

The raw diff is dominated by a single structural fact — the UK is absent from
NUTS 2024 — which would otherwise hide how small the genuine restructuring is.
Kosovo (XK) appears only in 2024.

In [ ]:
nuts3_24 = gpd.read_file(raw / "NUTS_RG_01M_2024_3035_LEVL_3.geojson")

ids_21, ids_24 = set(nuts3["NUTS_ID"]), set(nuts3_24["NUTS_ID"])
gone, new = ids_21 - ids_24, ids_24 - ids_21

print(f"2021: {len(ids_21)}   2024: {len(ids_24)}   stable: {len(ids_21 & ids_24)}\n")

dropped_cntr = {c[:2] for c in gone} - {c[:2] for c in ids_24}
added_cntr = {c[:2] for c in new} - {c[:2] for c in ids_21}
print(f"countries only in 2021: {sorted(dropped_cntr)}")
print(f"countries only in 2024: {sorted(added_cntr)}\n")

restructured = (
    pd.DataFrame(
        {
            "gone": pd.Series(sorted(gone)).str[:2].value_counts(),
            "new": pd.Series(sorted(new)).str[:2].value_counts(),
        }
    )
    .fillna(0)
    .astype(int)
)
real = restructured.drop(index=dropped_cntr | added_cntr, errors="ignore")

print("genuine restructuring (countries present in both vintages):")
print(real.to_string() if len(real) else "none")

Code-set differences alone cannot tell a rename from a split from a
coincidental code reuse — that is what the release notes are for. This is only
a sizing exercise for the crosswalk.

In [ ]:
for name in ["nuts-2021-release-notes.txt", "nuts-2024-release-notes.txt"]:
    print(f"--- {name} ---")
    print((raw / name).read_text(encoding="utf-8-sig").strip())
print()

## Population grid

GISCO 1 km grid v1.4 (B1-07). The NUTS-3 code ships on the cell, so the
population-weighted centroid is a groupby rather than a spatial join.

Parquet needs `pyarrow` in the `dev` extra: `uv add --optional dev "pyarrow>=17"`.

In [ ]:
grid = pd.read_parquet(next(raw.glob("*grid*1km*.parquet")))

print(f"{len(grid):,} cells")
print(grid.dtypes.to_string(), "\n")
grid.head(3)

### Cell attribution

Three traps, all of which silently produce plausible output if missed:

- unassigned cells carry an **empty string**, not `NaN`, so `isna()` finds
  nothing;
- codes are attributed to every cell intersecting **or lying within roughly
  1.5 km** of a region, so border cells carry **hyphen-joined** codes
  (`AL011-AL012`, and `CNTR_ID` likewise as `BA-HR-RS`). Splitting on `-` is
  unambiguous because NUTS codes contain none;
- `X_LLC` / `Y_LLC` are the cell's **lower-left corner**, not its centre —
  `calib/etl/step1_zones.py` adds 500 m to each before weighting.

In [ ]:
pop_col, nuts_col = "TOT_P_2021", "NUTS2021_3"
SEP = "-"  # NUTS codes contain no hyphen; border cells join codes with it

assigned = grid[nuts_col].ne("")
sub = grid.loc[assigned, [nuts_col, pop_col]].assign(
    code=lambda df: df[nuts_col].str.split(SEP)
)
n_codes = sub["code"].str.len()

print(f"cells:      {len(grid):,}")
print(f"unassigned: {(~assigned).sum():,} ({(~assigned).mean():.1%})")
print(f"codes per assigned cell: {n_codes.value_counts().sort_index().to_dict()}")

multi_pop = sub.loc[n_codes > 1, pop_col].sum()
print(f"population in multi-code cells: {multi_pop:,.0f}")
print(f"  ({multi_pop / grid[pop_col].sum():.2%} of total)")

### Zone coverage

⚠️ Coverage only: a border cell's population is counted once per code it
carries, so these totals double-count. `calib/etl/step1_zones.py` resolves each
multi-code cell to a single zone by point-in-polygon on the cell centre, and
counts its population once.

That also makes the uncovered count below a **pre-resolution** figure. It
rises slightly after resolution, because a few zones in uncovered countries
show population here only through a border cell shared with a covered
neighbour — LI and ME are absent from the list for exactly that reason.

In [ ]:
zone_pop = sub.explode("code").groupby("code")[pop_col].sum()

zones = set(nuts3["NUTS_ID"])
no_cells = zones - set(zone_pop.index)
no_pop = {z for z in zones & set(zone_pop.index) if zone_pop[z] == 0}

print(f"weightable zones: {len(zones) - len(no_cells) - len(no_pop)} / {len(zones)}\n")
print(f"no cells ({len(no_cells)}): {sorted(no_cells)}")
print(f"cells but no population ({len(no_pop)}):")
print(f"  by country: {pd.Series(sorted(no_pop)).str[:2].value_counts().to_dict()}")

### Findings

Population coverage is **EU-27 + CH + NO only** — the 2021 census round did
not cover AL, IS, LI, ME, MK, RS, TR or UK. Roughly 1 225 of 1 514 zones are
weightable, and every Tier 1 zone is among them, so the gap is a documented
limitation rather than a blocker and no fallback data is fetched.

Four outcomes, not two:

| Bucket | Zones | Treatment |
|---|---|---|
| Weightable (EU-27 + CH + NO, continental) | ~1 225 | population-weighted |
| Outside the 2021 census round | ~282 | geometric |
| Outside the grid extent — the French DOM, which EPSG:3035 does not reach | 5 | geometric, `tier = 2` |
| No cells at all — Jan Mayen, Svalbard | 2 | geometric, `tier = 2` |

`centroid_source` therefore stays binary and a separate
`centroid_fallback_reason` carries the diagnosis (D8).

In [ ]:
dom = sorted(z for z in no_pop if z.startswith("FR"))
print(
    nuts3[nuts3["NUTS_ID"].isin(dom)][["NUTS_ID", "NAME_LATN"]].to_string(index=False)
)

## GHS-POP — closing the coverage gaps

The GEOSTAT grid covers EU-27 + CH + NO only (F14). With Tier 2 routing
approaching, the uncovered zones need a population weighting from somewhere,
and GHS-POP R2023A is the one source that closes every gap at once: it is
global, so it reaches the UK, the candidate countries, Iceland, Liechtenstein
**and** the French DOM, which no European-projection product can.

It is modelled rather than counted — census and administrative totals
disaggregated onto a grid using built-up distribution, density and
classification. So it is a fallback, never a replacement: GEOSTAT wins
wherever it exists, and `centroid_source` records which was used.

In [ ]:
import numpy as np
import rasterio
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds
from rasterio.windows import transform as window_transform

ghs_path = next(raw.glob("GHS_POP_*.tif"))
ghs = rasterio.open(ghs_path)

# Filename encodes epoch and resolution: GHS_POP_E<epoch>_GLOBE_R2023A_<crs>_<res>_V1_0
parts = ghs_path.stem.split("_")
epoch, resolution_m = int(parts[2][1:]), int(parts[6])

print(f"file:       {ghs_path.name}")
print(f"epoch:      {epoch}")
print(f"resolution: {resolution_m} m")
print(f"CRS:        {ghs.crs}")
print(f"shape:      {ghs.shape[0]:,} x {ghs.shape[1]:,} = {np.prod(ghs.shape):,} cells")
print(f"nodata:     {ghs.nodata}   dtype: {ghs.dtypes[0]}")

if epoch != 2020:
    print(f"\n[!] epoch {epoch} is a projection, not an observed year.")
    print("    GEOSTAT is the 2021 census round; mixing vintages biases the")
    print("    fallback zones systematically. See IMPLEMENTATION_DOCU.md F17.")

⚠️ Never read the whole band — a global grid at this resolution is far too
large to hold in memory. Every cell below reads a per-zone window.

Rasterising the zone polygon onto its own window also sidesteps the
border-cell problem entirely: each cell falls in exactly one zone by
construction, unlike GEOSTAT's pre-attributed codes.

In [ ]:
def ghs_centroid(geom, buffer_m: float = 2_000.0) -> tuple[float, float, float] | None:
    """Population-weighted centroid of one zone, in the raster's CRS.

    Returns (x, y, population), or None where the zone holds no population.
    """
    minx, miny, maxx, maxy = geom.bounds
    win = from_bounds(
        minx - buffer_m,
        miny - buffer_m,
        maxx + buffer_m,
        maxy + buffer_m,
        ghs.transform,
    )
    arr = ghs.read(1, window=win, boundless=True, fill_value=0)
    if arr.size == 0:
        return None

    wt = window_transform(win, ghs.transform)
    inside = geometry_mask([geom], arr.shape, wt, invert=True)
    pop = np.where(inside & (arr > 0), arr, 0).astype("float64")

    total = pop.sum()
    if total <= 0:
        return None

    rows, cols = np.indices(arr.shape)
    xs, ys = rasterio.transform.xy(wt, rows, cols)
    return (
        float(np.average(np.asarray(xs), weights=pop)),
        float(np.average(np.asarray(ys), weights=pop)),
        float(total),
    )


zones_moll = nuts3.to_crs(ghs.crs).set_index("NUTS_ID")
print(f"reprojected {len(zones_moll)} zones to {ghs.crs}")

### Do the gaps close?

National totals are the check that the epoch and CRS assumptions hold. The UK
should land near 67 M and Türkiye near 85 M for an observed year; a projected
epoch runs a few percent above.

In [ ]:
gaps = sorted(no_pop | no_cells)

rows = []
for zid in gaps:
    res = ghs_centroid(zones_moll.loc[zid, "geometry"])
    rows.append(
        {
            "nuts_id": zid,
            "country": zid[:2],
            "resolved": res is not None,
            "pop": res[2] if res else 0.0,
        }
    )
gap_df = pd.DataFrame(rows)

print(
    f"resolved {gap_df['resolved'].sum()} / {len(gap_df)} previously uncovered zones\n"
)
print(
    gap_df.groupby("country")
    .agg(
        zones=("nuts_id", "size"),
        resolved=("resolved", "sum"),
        pop_m=("pop", lambda s: round(s.sum() / 1e6, 2)),
    )
    .sort_values("pop_m", ascending=False)
    .to_string()
)
print(f"\nstill unresolved: {gap_df.loc[~gap_df['resolved'], 'nuts_id'].tolist()}")

### How much error does the fallback carry?

The zones that need GHS-POP are exactly the zones with nothing to check it
against — so the check has to happen where **both** sources exist. Computing
each centroid twice across a sample of covered zones gives an empirical
displacement distribution, which is the only honest basis for relying on the
fallback for 200-odd UK zones.

The GEOSTAT reference uses single-code cells only, so no border-cell
resolution rule is needed for the comparison, and cell centres are lower-left
plus half a cell.

In [ ]:
SAMPLE_N = 250
rng = np.random.default_rng(0)
covered = sorted(set(nuts3["NUTS_ID"]) - set(gaps))
sample = set(rng.choice(covered, size=SAMPLE_N, replace=False))

single = grid.loc[assigned & n_codes.eq(1).reindex(grid.index, fill_value=False)]
geo = single.loc[single[nuts_col].isin(sample) & single[pop_col].gt(0)]

ref = (
    geo.assign(x=geo["X_LLC"] + 500, y=geo["Y_LLC"] + 500)
    .groupby(nuts_col)
    .apply(
        lambda d: pd.Series(
            {
                "x": np.average(d["x"], weights=d[pop_col]),
                "y": np.average(d["y"], weights=d[pop_col]),
            }
        ),
        include_groups=False,
    )
)
print(f"GEOSTAT centroids for {len(ref)} sampled zones")

In [ ]:
alt = {z: ghs_centroid(zones_moll.loc[z, "geometry"]) for z in ref.index}
alt = {z: v for z, v in alt.items() if v is not None}

alt_pts = gpd.GeoSeries(
    gpd.points_from_xy([v[0] for v in alt.values()], [v[1] for v in alt.values()]),
    index=list(alt),
    crs=ghs.crs,
).to_crs(3035)
ref_pts = gpd.GeoSeries(
    gpd.points_from_xy(ref["x"], ref["y"]), index=ref.index, crs=3035
).loc[alt_pts.index]

disp = (alt_pts.distance(ref_pts) / 1000).sort_values()

print(disp.describe().round(2).to_string())
print(f"\nmedian {disp.median():.1f} km   90th pct {disp.quantile(0.9):.1f} km")

Read against the access-time cap rather than against zero. A median inside a
few kilometres is well below the resolution a 90-minute cap can distinguish,
so the fallback is sound. A long tail is expected and mostly benign — it
should fall in large rural zones, which are also where the geometric
alternative is worst, so GHS-POP still wins there. A compact urban zone in the
tail would mean something structural is wrong.

In [ ]:
worst = disp.tail(8).index.tolist()
print(
    nuts3[nuts3["NUTS_ID"].isin(worst)][
        ["NUTS_ID", "NAME_LATN", "area_km2", "URBN_TYPE"]
    ]
    .assign(displacement_km=lambda df: df["NUTS_ID"].map(disp).round(1))
    .sort_values("displacement_km", ascending=False)
    .to_string(index=False)
)

urban = nuts3.set_index("NUTS_ID").loc[worst, "URBN_TYPE"]
print(f"\nurban types in the tail: {urban.value_counts().sort_index().to_dict()}")
print("(1 = predominantly urban, 2 = intermediate, 3 = predominantly rural)")

### ⚠️ The source decision cannot be taken here

Below, zones outside the gap set are labelled `pop_grid` — and for a handful
that is wrong. Some zones in countries the census grid does not cover still
show census population, drawn entirely from **border cells shared with a
covered neighbour**. They never entered the gap set, so they look covered.

Once the ETL assigns each cell to exactly one zone by point-in-polygon, that
population goes to the neighbour and these zones drop to zero. So
`calib/etl/step1_zones.py` resolves cells first and decides the source
afterwards; taking the assignment from this cell would leave them weighted on
their neighbours' population, and nothing would fail.

The cell below therefore prints a *pre-resolution* picture, and the countries
it exposes are the ones to check against the ETL's own report.

In [ ]:
source = pd.Series("pop_grid", index=nuts3["NUTS_ID"], name="centroid_source")
source.loc[sorted(set(gap_df.loc[gap_df["resolved"], "nuts_id"]))] = "ghs_pop"
source.loc[sorted(set(gap_df.loc[~gap_df["resolved"], "nuts_id"]))] = "geometric"

print(source.value_counts().to_string(), "\n")
by_country = (
    pd.DataFrame({"src": source, "cntr": source.index.str[:2]})
    .groupby(["cntr", "src"])
    .size()
    .unstack(fill_value=0)
)
print(by_country[by_country.get("ghs_pop", 0) > 0].to_string())

# Countries with no census population of their own: any zone of theirs shown
# as pop_grid is borrowing it across a border, and will move to ghs_pop once
# the ETL resolves cells to single zones.
uncovered_countries = {z[:2] for z in gaps}
borrowed = sorted(
    z for z in source.index if z[:2] in uncovered_countries and source[z] == "pop_grid"
)
print(f"\nzones on borrowed border-cell population ({len(borrowed)}):")
print(f"  {pd.Series(borrowed).str[:2].value_counts().to_dict()}")

### Where this ended up

⚠️ The figures in this notebook are the **exploration-time** picture and are
superseded by the ETL. Two things changed afterwards:

- Census coverage turned out to be a **country-level** property, not a
  cell-level one. Resolving border cells to single zones settles which zone a
  cell belongs to, but not who counted the people in it — a cell straddling
  the Irish border whose centre falls on the UK side is a UK cell holding
  population only Ireland counted. So `calib/etl/step1_zones.py` gates the
  census grid on national coverage, and **315** zones fall back to GHS-POP,
  not the 289 found here (`IMPLEMENTATION_DOCU.md` F26).
- **Liechtenstein is census-covered.** It looked uncovered here only because a
  per-country table denominated in millions rounded its 37 328 people to
  `0.00`. Coverage is 30 countries, not 29 (F14, corrected).

Final: 1 514 zones — 1 199 census grid, 315 GHS-POP, none geometric. This
notebook is kept as the record of how those numbers were reached, wrong turns
included; `calib/out/01_zones.parquet` is the authority.